In [2]:
# !pip install langchain langchain-community sentence-transformers faiss-cpu rank_bm25 pypdf requests

In [3]:
import os
import re
import requests
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi

In [5]:
PDF_FOLDER = "pdfs"  # create this folder and put the PDFs here
print(PDF_FOLDER)

pdfs


In [7]:
# Loads the PDF from the "pdfs" folder
def load_pdfs(folder):
    documents = []
    
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            reader = PdfReader(os.path.join(folder, file))
            text = ""
            for page in reader.pages:
                text += page.extract_text() or ""
            
            documents.append({
                "text": text,
                "source": file
            })
    
    return documents

In [8]:
# Chunks the pdf doucment
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks

In [9]:
# Embedding model = all-MiniLM-L6-v2
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\AI-Dev\miniconda3\envs\gpuenv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\AI-Dev\huggingface_cache\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# Define Vector Store class
class VectorStore:
    def __init__(self):
        self.texts = []
        self.embeddings = None
        self.index = None

    def add(self, chunks):
        self.texts.extend(chunks)
        embeddings = embedding_model.encode(chunks)

        if self.embeddings is None:
            self.embeddings = embeddings
        else:
            self.embeddings = list(self.embeddings) + list(embeddings)

        self.index = faiss.IndexFlatL2(len(embeddings[0]))
        self.index.add(self.embeddings)

    def search(self, query, k=5):
        query_vec = embedding_model.encode([query])
        distances, indices = self.index.search(query_vec, k)

        return [self.texts[i] for i in indices[0]]

In [11]:
# Define BM25 Keyword search class
class KeywordStore:
    def __init__(self, chunks):
        self.tokenized = [chunk.split() for chunk in chunks]
        self.bm25 = BM25Okapi(self.tokenized)
        self.chunks = chunks

    def search(self, query, k=5):
        tokenized_query = query.split()
        scores = self.bm25.get_scores(tokenized_query)
        top_k = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return [self.chunks[i] for i in top_k]

In [12]:
# Define the hybrid search using vector store and keyword search
def hybrid_search(query, vector_store, keyword_store, k=5):
    semantic_results = vector_store.search(query, k)
    keyword_results = keyword_store.search(query, k)

    combined = list(set(semantic_results + keyword_results))
    return combined[:k]

In [13]:
# Function for Ollama LLM response using llama3
def ollama_llm(prompt):
    import requests
    
    # url = "http://localhost:11434/api/generate"
    url = "http://host.docker.internal:11434/api/generate"
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False
    }
    
    response = requests.post(url, json=payload)
    return response.json()["response"]

In [14]:
# Function to compare the 2 versions of the document
def compare_clauses(docA, docB):
    prompt = f"""
Compare the following two legal clauses:

Clause A:
{docA}

Clause B:
{docB}

Highlight key differences, risks, and implications.
"""
    return ollama_llm(prompt)

In [15]:
# Define the Agent router
def agent_router(query, vector_store, keyword_store):
    
    results = hybrid_search(query, vector_store, keyword_store)

    context = "\n\n".join(results)

    if "compare" in query.lower():
        if len(results) >= 2:
            return compare_clauses(results[0], results[1])

    prompt = f"""
You are a legal assistant.

Context:
{context}

Question:
{query}

Answer clearly and accurately based only on the context.
"""
    
    return ollama_llm(prompt)

In [22]:
# Create an ingestion pipeline to load pdfs from "pdfs" folder. 
# Create chunks for each pdf and store it in the vector store and keyword store.
import os
print(os.getcwd())

if not os.path.exists(PDF_FOLDER):
    os.makedirs(PDF_FOLDER)
    print(f"{PDF_FOLDER} folder created")

documents = load_pdfs(PDF_FOLDER)

all_chunks = []
for doc in documents:
    chunks = chunk_text(doc["text"])
    all_chunks.extend(chunks)

vector_store = VectorStore()
vector_store.add(all_chunks)

keyword_store = KeywordStore(all_chunks)

print("Ingestion complete. Total chunks:", len(all_chunks))

d:\Sanjeevv\TekPro_AI\Legal_Chatbot_v1\SourceCode
Ingestion complete. Total chunks: 811


In [23]:
# Test query
query = "What is indemnity clause in this contract?"

response = agent_router(query, vector_store, keyword_store)

print(response)

Based on the provided context, the indemnity clause in this contract states that "The party of the first part agrees to indemnify and hold harmless the second party." This means that Warner Inc (the party of the first part) is agreeing to take responsibility for any potential damages or liabilities that may arise and protect Castillo Group (the second party) from any legal consequences.


In [24]:
# Compare indemnity clauses
query = "Compare indemnity clauses in the documents"

response = agent_router(query, vector_store, keyword_store)

print(response)

Based on the two legal clauses provided (Clause A and Clause B), here are the key differences, risks, and implications:

**Key Differences:**

1. **Agreement Title:** The most obvious difference is the title of each agreement. Clause A has a more vague title ("Diverse next generation paradigm"), while Clause B has a more specific title ("Focused content-based pricing structure").
2. **Parties Involved:** The parties involved in each clause are different. In Clause A, it's Williams-Wilson and Weiss-Walker, while in Clause B, it's Marquez-Lambert and Williams-Richard.
3. **Indemnification Provisions:** Both clauses have indemnification provisions, but they differ slightly. Clause A has a broader indemnification clause that applies to "the second party," while Clause B has a more specific indemnification provision that only applies to the "second party."
4. **Rent Payment Terms:** Clause A mentions rent payment terms, stating that the lessee must pay rent before the 5th of each month. Cla